# Module 1 - First Call

In this notebook you make your first Amazon Bedrock call with the Converse API, and turn it into a simple travel chat assistant for AnyCompany.

## Contents

1. [Introduction](#intro)
2. [Setup](#setup)
3. [Your first Converse call](#first-call)
4. [Give the assistant a role with a system prompt](#system-prompt)
5. [Control the response with inference parameters](#inference-config)
6. [Stream the response with ConverseStream](#streaming)
7. [Hold a conversation across turns](#multi-turn)
8. [The same call with provider-compatible APIs](#compatible-apis)
9. [Try it](#try-it)
10. [Conclusion and key takeaways](#conclusion)
11. [Next steps](#next-steps)

## 1. Introduction <a id="intro"></a>

AnyCompany's travel team answers the same questions every day: how to book, what a per diem covers, what to do when a flight is cancelled. Over this workshop you build an assistant that takes those questions off their plate. In this module you build its first version: a chat assistant that talks to a model on Amazon Bedrock.

You use the [Converse API](https://docs.aws.amazon.com/bedrock/latest/userguide/conversation-inference.html), Amazon Bedrock's single API for conversations with any supported model. You write the request once, and the same code works whether the model is from Anthropic, Amazon, Mistral or another provider. That matters in the next module, when you compare models.

By the end of this module, your assistant can:

| Capability | API feature |
|:--|:--|
| Answer a question | `converse` request and response |
| Act as AnyCompany's travel assistant | System prompt |
| Keep answers within a length budget | `inferenceConfig` |
| Show the answer as it is generated | `converse_stream` |
| Remember earlier messages in the conversation | Conversation history |

## 2. Setup <a id="setup"></a>

Run the cell below to create the Amazon Bedrock client and choose the model for this module.

In [ ]:
import boto3

# The AWS Region from your AWS configuration (or the AWS_REGION environment variable).
REGION = boto3.session.Session().region_name

bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)

# Claude Sonnet 5.5, called through a cross-Region inference profile.
# The "global." prefix lets Amazon Bedrock route each request to any AWS Region with
# capacity. Use "us." instead to keep requests within US Regions.
MODEL_ID = "global.anthropic.claude-sonnet-5-5"

print(f"Region: {REGION}")
print(f"Model:  {MODEL_ID}")

## 3. Your first Converse call <a id="first-call"></a>

A Converse request has two required parts:

- `modelId`: which model to call.
- `messages`: the conversation so far. Each message has a `role` (`user` or `assistant`) and a list of `content` blocks. A content block can hold text, an image or a document; here you send one text block.

Run the cell below to ask the model a travel question.

In [ ]:
response = bedrock_runtime.converse(
    modelId=MODEL_ID,
    messages=[
        {
            "role": "user",
            "content": [{"text": "What is a per diem, in one or two sentences?"}],
        }
    ],
)

print(response["output"]["message"]["content"][0]["text"])

The reply is more than text. The response also tells you why the model stopped, how many tokens it used and how long the call took. You use these numbers later to compare models and to monitor cost.

Run the cell below to look at them.

In [ ]:
print("Stop reason:   ", response["stopReason"])
print("Input tokens:  ", response["usage"]["inputTokens"])
print("Output tokens: ", response["usage"]["outputTokens"])
print("Latency (ms):  ", response["metrics"]["latencyMs"])

> **Note:** `content` is a list because a response can contain more than one block. Models that reason before answering can return reasoning blocks before the text. The helper below collects only the text blocks, so your code keeps working whatever the model returns. You use it for the rest of the workshop.

In [ ]:
def get_text(response):
    """Return the text blocks of a Converse response, joined together."""
    blocks = response["output"]["message"]["content"]
    return "".join(block["text"] for block in blocks if "text" in block)


print(get_text(response))

## 4. Give the assistant a role with a system prompt <a id="system-prompt"></a>

Right now the model answers like a general-purpose assistant. A **system prompt** sets its role, tone and rules for the whole conversation. You pass it in the `system` parameter, separate from the user's messages.

Run the cell below to turn the model into AnyCompany's travel assistant.

In [ ]:
SYSTEM_PROMPT = [
    {
        "text": (
            "You are the travel assistant for AnyCompany, helping employees plan and "
            "manage business travel. Be friendly and concise: answer in three sentences "
            "or fewer unless the employee asks for detail. If you are not sure about an "
            "AnyCompany-specific rule, say so and suggest contacting the travel team."
        )
    }
]

response = bedrock_runtime.converse(
    modelId=MODEL_ID,
    system=SYSTEM_PROMPT,
    messages=[
        {
            "role": "user",
            "content": [{"text": "My flight to Denver was cancelled. What should I do?"}],
        }
    ],
)

print(get_text(response))

## 5. Control the response with inference parameters <a id="inference-config"></a>

The `inferenceConfig` parameter controls how the model generates its answer. The most common settings are:

| Parameter | What it does |
|:--|:--|
| `maxTokens` | The maximum number of tokens in the response. Use it to cap cost and length. |
| `stopSequences` | Text that makes the model stop as soon as it generates it. |
| `temperature`, `topP` | How random the output is. **Not every model accepts these.** |

Run the cell below with a deliberately small `maxTokens`. The stop reason changes from `end_turn` to `max_tokens`, which tells you the response is incomplete. Look at the content blocks too: there may be no text at all.

In [ ]:
PACKING_QUESTION = [
    {"role": "user", "content": [{"text": "List five things to pack for a business trip."}]}
]

response = bedrock_runtime.converse(
    modelId=MODEL_ID,
    system=SYSTEM_PROMPT,
    messages=PACKING_QUESTION,
    inferenceConfig={"maxTokens": 30},
)

print("Stop reason:   ", response["stopReason"])
print("Content blocks:", [list(block)[0] for block in response["output"]["message"]["content"]])
print("Text:          ", repr(get_text(response)))

Claude Sonnet 5.5 uses **adaptive thinking**: it decides how much to reason before it answers, and the reasoning counts toward `maxTokens`. With a budget of 30 tokens, the reasoning can use all of it, leaving a `reasoningContent` block and no text.

You can tell the model how much effort to spend. Settings that are specific to one model go in `additionalModelRequestFields`, which Amazon Bedrock passes through to the model unchanged. Run the cell below with low effort: the model now spends the budget on the answer, which is still cut off at 30 tokens.

In [ ]:
response = bedrock_runtime.converse(
    modelId=MODEL_ID,
    system=SYSTEM_PROMPT,
    messages=PACKING_QUESTION,
    inferenceConfig={"maxTokens": 30},
    additionalModelRequestFields={"output_config": {"effort": "low"}},
)

print("Stop reason:", response["stopReason"])
print(get_text(response))

> **Important:** Set `maxTokens` with room for reasoning as well as the answer, and check for the `max_tokens` stop reason in your code.

Inference parameters are model-specific. Claude Sonnet 5.5 manages its own sampling, so it rejects `temperature` and `topP`, while other models on Amazon Bedrock accept them. When a parameter is not supported, Amazon Bedrock returns a `ValidationException` that names it.

Run the cell below to see the error.

In [ ]:
from botocore.exceptions import ClientError

try:
    bedrock_runtime.converse(
        modelId=MODEL_ID,
        messages=[{"role": "user", "content": [{"text": "Hello"}]}],
        inferenceConfig={"temperature": 0.2},
    )
except ClientError as error:
    print(error.response["Error"]["Code"])
    print(error.response["Error"]["Message"])

> **Note:** Each model's [model card](https://docs.aws.amazon.com/bedrock/latest/userguide/model-cards.html) lists the parameters and features it supports. Check it when you adopt a new model.

## 6. Stream the response with ConverseStream <a id="streaming"></a>

With `converse`, you wait for the whole answer before you see anything. For a chat assistant, it feels faster to show the answer as it is generated. [`converse_stream`](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_runtime_ConverseStream.html) takes the same request, but returns a stream of events:

| Event | Contains |
|:--|:--|
| `messageStart` | The start of the response |
| `contentBlockDelta` | The next piece of text |
| `contentBlockStop`, `messageStop` | The end of a block and of the response, with the stop reason |
| `metadata` | Token usage and latency |

Run the cell below to print the answer piece by piece as it arrives.

In [ ]:
response = bedrock_runtime.converse_stream(
    modelId=MODEL_ID,
    system=SYSTEM_PROMPT,
    messages=[
        {
            "role": "user",
            "content": [{"text": "Give me three tips for a smooth airport security check."}],
        }
    ],
    inferenceConfig={"maxTokens": 400},
)

for event in response["stream"]:
    if "contentBlockDelta" in event:
        delta = event["contentBlockDelta"]["delta"]
        if "text" in delta:
            print(delta["text"], end="", flush=True)
    elif "metadata" in event:
        usage = event["metadata"]["usage"]
        print(f"\n\n[{usage['inputTokens']} input tokens, {usage['outputTokens']} output tokens]")

## 7. Hold a conversation across turns <a id="multi-turn"></a>

The model does not remember previous calls. Each request is independent, so to hold a conversation you send the history every time: the user's messages and the model's replies, in order.

The `chat` function below does this for you. It keeps a `history` list, adds each question and answer to it, and sends the whole list with every request. This function is the core of your travel assistant, and you extend it in later modules.

In [ ]:
def chat(history, user_message, max_tokens=500):
    """Send a message with the conversation so far, and record the reply in history."""
    history.append({"role": "user", "content": [{"text": user_message}]})
    response = bedrock_runtime.converse(
        modelId=MODEL_ID,
        system=SYSTEM_PROMPT,
        messages=history,
        inferenceConfig={"maxTokens": max_tokens},
    )
    history.append(response["output"]["message"])
    return get_text(response)


history = []
print(chat(history, "I'm flying to Chicago next Tuesday for a two-day client meeting."))

Now ask a follow-up question that only makes sense with the earlier message. The model can answer because `chat` sends the whole history.

In [ ]:
print(chat(history, "Which airport should I fly into, and how do I get downtown from there?"))
print(f"\nMessages in history: {len(history)}")

> **Note:** Every message in the history counts as input tokens on every call, so long conversations cost more per turn. Production applications usually trim or summarize old messages.

## 8. The same call with provider-compatible APIs <a id="compatible-apis"></a>

Converse is not the only way to call models on Amazon Bedrock. The `bedrock-runtime` endpoint also accepts provider-compatible APIs, such as Anthropic's Messages API and OpenAI's Responses API, so code written for a provider's own SDK runs on Amazon Bedrock with a different base URL. Each model's [model card](https://docs.aws.amazon.com/bedrock/latest/userguide/model-cards.html) lists the APIs it supports.

You try two of them below, with the same travel question.

### Anthropic Messages API

Run the cell below to send the question to Claude Sonnet 5.5 with the Anthropic SDK. It authenticates with a short-term Amazon Bedrock API key, generated from your current AWS credentials.

In [ ]:
from anthropic import Anthropic
from aws_bedrock_token_generator import provide_token

anthropic_client = Anthropic(
    base_url=f"https://bedrock-runtime.{REGION}.amazonaws.com/anthropic",
    api_key=provide_token(region=REGION),
)

message = anthropic_client.messages.create(
    model=MODEL_ID,
    max_tokens=300,
    system=SYSTEM_PROMPT[0]["text"],
    messages=[{"role": "user", "content": "What is a per diem, in one or two sentences?"}],
)

print(message.content[-1].text)

### OpenAI Responses API

Claude Sonnet 5.5 does not support the OpenAI APIs, so for this example you use an OpenAI model: **GPT-6 Luna**, OpenAI's low-cost model for everyday tasks. The code is what you would write for OpenAI's own service; only the base URL and the API key point to Amazon Bedrock.

Run the cell below to send the same question with the OpenAI SDK.

In [ ]:
from openai import OpenAI

openai_client = OpenAI(
    base_url=f"https://bedrock-runtime.{REGION}.amazonaws.com/openai/v1",
    api_key=provide_token(region=REGION),
)

response = openai_client.responses.create(
    # GPT-6 Luna, called through the US cross-Region inference profile.
    model="us.openai.gpt-6-luna",
    instructions=SYSTEM_PROMPT[0]["text"],
    input="What is a per diem, in one or two sentences?",
    max_output_tokens=500,
)

print(response.output_text)

> **Note:** OpenAI GPT models on Amazon Bedrock also need permission on your account's default project: `bedrock:InvokeModel` on `arn:aws:bedrock:<region>:<account-id>:project/default`, in addition to the model. At AWS events, your account already has it.

All three calls have the same ingredients (model, system prompt, question, a token limit) in different shapes. This workshop uses Converse from here on, because the features in later modules, such as Guardrails and service tiers, work with it across every model you use. With the Responses API, for example, Guardrails are not available.

> **Note:** Amazon Bedrock also has a second endpoint, `bedrock-mantle`, which serves the provider-compatible APIs and features such as background inference. It does not support Converse or Guardrails, so for new applications AWS recommends `bedrock-runtime`, as used in this workshop.

## 9. Try it <a id="try-it"></a>

1. Change `SYSTEM_PROMPT` so the assistant answers in bullet points, then run the `chat` cells again.
2. Start a new conversation with `history = []` and ask: "What is AnyCompany's nightly hotel limit in Chicago?" Notice that the model has no way to know AnyCompany's actual policy. You fix this in Module 3.

In [ ]:
history = []
print(chat(history, "What is AnyCompany's nightly hotel limit in Chicago?"))

## 10. Conclusion and key takeaways <a id="conclusion"></a>

- The **Converse API** gives you one request format for conversations with any supported model. Code written for the **Anthropic** or **OpenAI** SDKs also runs on Amazon Bedrock, by changing the base URL and API key.
- A **system prompt** sets the assistant's role and rules; **inferenceConfig** limits length and stopping. Supported parameters vary by model, and on reasoning models `maxTokens` covers thinking as well as the answer.
- **ConverseStream** returns the answer as it is generated, which makes chat feel faster.
- Models are **stateless**: you hold a conversation by sending the history with every request.
- Every response reports **token usage and latency**, the raw material for choosing models and monitoring cost.

## 11. Next steps <a id="next-steps"></a>

Your assistant can now chat. In [Module 2 - Choosing a Model](02_choosing_a_model.ipynb), AnyCompany's finance team asks for a second capability, checking expense reports, and you pick the right model for each task by comparing quality, latency and cost.